# Let's Create a Knowledge Graph

We'll try to keep things tractable, while using real-world data.

Let's start by fetching the RAF/MAP kinase cascade pathway from the Reactome database.

We'll fetch all the 'participants' or diverse entities that are known to participate in the pathway.

Later we'll subset this.

In [ ]:
import requests
import json
from pathlib import Path
from tqdm.notebook import tqdm

# Reactome pathway (example: RAF/MAP kinase cascade / R-HSA-5673001)
PATHWAY_ID = "R-HSA-5673001"

# Reactome API
url = f"https://reactome.org/ContentService/data/participants/{PATHWAY_ID}"

print("Fetching Reactome participants...")

data = requests.get(url).json()

Path("data_raw").mkdir(exist_ok=True)

with open("data_raw/reactome_participants.json", "w") as f:
    json.dump(data, f, indent=2)

print("Saved to data_raw/reactome_participants.json")
print("Participants:", len(data))

## Now that we have the participants downloaded into a local file json format, let's have a look at what's inside.

You'll notice CandidateSets containing many other entities.  
We'll subset this for simplicity.

In [ ]:
with open("data_raw/reactome_participants.json") as f:
    data=json.load(f)

data[:2]

## Let's extract a list of only the protein IDs

We'll save this to file and make use of it later.

In [ ]:
import json
from pathlib import Path

with open("data_raw/reactome_participants.json") as f:
    participants = json.load(f)

proteins = {}

for p in participants:
    for ref in p.get("refEntities", []):
        st_id = ref.get("stId", "")
        identifier = ref.get("identifier")
        name = ref.get("displayName")

        if st_id.startswith("uniprot:") and identifier:
            proteins[identifier] = name

protein_list = [
    {
        "uniprot_accession": acc,
        "name": name
    }
    for acc, name in proteins.items()
]

protein_list = sorted(protein_list, key=lambda x: x["uniprot_accession"])

Path("data_processed").mkdir(exist_ok=True)

with open("data_processed/proteins.json", "w") as f:
    json.dump(protein_list, f, indent=2)

print("Proteins extracted:", len(protein_list))
print("Saved to data_processed/proteins.json")

## Let's have a look at the file content.

We will use this cleaned up file of protein IDs for filtering later.

Focusing on proteins will make the demo easier to process computationally and mentally.

In [ ]:
with open("data_processed/proteins.json") as f:
    data=json.load(f)

data[:20]

## Let's use the list of proteins to retrieve interactors from the Chembl database.

We'll create drug:target records and save these also.

In [ ]:
import json
from pathlib import Path
import requests

# load clean protein list previous step
with open("data_processed/proteins.json") as f:
    proteins = json.load(f)

uniprot_ids = sorted({p["uniprot_accession"] for p in proteins})

print("Unique UniProt accessions:", len(uniprot_ids))
print(uniprot_ids[:10])

results = []
seen = set()

print("Querying ChEMBL using UniProt accessions.  This will take ~10 minutes...")

for accession in tqdm(uniprot_ids):

    target_url = (
        "https://www.ebi.ac.uk/chembl/api/data/target.json"
        f"?target_components__accession={accession}"
        "&target_type=SINGLE%20PROTEIN"
    )

    target_data = requests.get(target_url).json()

    for target in target_data.get("targets", []):
        target_id = target.get("target_chembl_id")
        target_name = target.get("pref_name")

        mech_url = (
            "https://www.ebi.ac.uk/chembl/api/data/mechanism.json"
            f"?target_chembl_id={target_id}"
        )

        mech_data = requests.get(mech_url).json()

        for mech in mech_data.get("mechanisms", []):
            mol_id = mech.get("molecule_chembl_id")

            if not mol_id:
                continue

            mol_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/{mol_id}.json"
            mol_data = requests.get(mol_url).json()

            drug_name = mol_data.get("pref_name")

            record_key = (
                accession,
                target_id,
                mol_id,
                mech.get("action_type"),
                mech.get("mechanism_of_action"),
            )

            if record_key in seen:
                continue

            seen.add(record_key)

            results.append({
                "uniprot_accession": accession,
                "target_chembl_id": target_id,
                "target_name": target_name,
                "molecule_chembl_id": mol_id,
                "drug_name": drug_name,
                "action_type": mech.get("action_type"),
                "mechanism_of_action": mech.get("mechanism_of_action"),
            })

Path("data_raw").mkdir(exist_ok=True)

with open("data_raw/chembl_mechanisms.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved to data_raw/chembl_mechanisms.json")
print("Drug-target mechanism records:", len(results))

## Now we'll have a look at the data we just saved.

In [ ]:
with open("data_raw/chembl_mechanisms.json") as f:
    data=json.load(f)

data[:5]

## Now we'll retrieve gene:disease associations 

We will query from the Open Targets database using its GraphQL API.

This will allow us to query disease associations for specific genes within our pathway later.


In [ ]:
GRAPHQL_URL = "https://api.platform.opentargets.org/api/v4/graphql"

with open("data_processed/proteins.json") as f:
    proteins = json.load(f)

search_query = """
query searchTarget($queryString: String!) {
  search(queryString: $queryString, entityNames: ["target"]) {
    hits {
      id
      entity
      object {
        ... on Target {
          id
          approvedSymbol
          approvedName
        }
      }
    }
  }
}
"""

assoc_query = """
query targetDiseases($ensemblId: String!) {
  target(ensemblId: $ensemblId) {
    id
    approvedSymbol
    associatedDiseases(page: {index: 0, size: 10}) {
      rows {
        disease {
          id
          name
        }
        score
      }
    }
  }
}
"""

results = []

print("Querying Open Targets disease associations using UniProt accessions.  This will take ~3 minutes...")

for protein in tqdm(proteins):
    accession = protein["uniprot_accession"]
    name = protein["name"]

    # name looks like: "UniProt:P00533 EGFR"
    symbol = name.split()[-1]

    #print("Searching Open Targets for", symbol)

    r = requests.post(
        GRAPHQL_URL,
        json={
            "query": search_query,
            "variables": {"queryString": symbol}
        }
    )
    search_data = r.json()

    hits = search_data.get("data", {}).get("search", {}).get("hits", [])

    if not hits:
        print("  No target found")
        continue

    hit = hits[0]
    target_obj = hit.get("object", {})
    ensembl_id = target_obj.get("id")
    approved_symbol = target_obj.get("approvedSymbol")

    if not ensembl_id:
        print("  No Ensembl ID found")
        continue

    #print("  Found target:", approved_symbol, ensembl_id)

    r = requests.post(
        GRAPHQL_URL,
        json={
            "query": assoc_query,
            "variables": {"ensemblId": ensembl_id}
        }
    )
    assoc_data = r.json()

    rows = (
        assoc_data.get("data", {})
        .get("target", {})
        .get("associatedDiseases", {})
        .get("rows", [])
    )

    for row in rows:
        disease = row.get("disease", {})
        disease_id = disease.get("id")
        disease_name = disease.get("name")
        score = row.get("score")

        results.append({
            "uniprot_accession": accession,
            "gene_symbol": approved_symbol,
            "target_id": ensembl_id,
            "disease_id": disease_id,
            "disease_name": disease_name,
            "association_score": score
        })

Path("data_raw").mkdir(exist_ok=True)

with open("data_raw/opentargets_diseases.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved to data_raw/opentargets_diseases.json")
print("Protein-disease records:", len(results))

## Now we have all our reference data - let's build a knowledge graph!

In [ ]:
import json
from pathlib import Path
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS
from rdflib.namespace import SKOS

# ----------------------------
# load input data
# ----------------------------

with open("data_processed/proteins.json") as f:
    proteins = json.load(f)

with open("data_raw/chembl_mechanisms.json") as f:
    chembl = json.load(f)

with open("data_raw/opentargets_diseases.json") as f:
    diseases = json.load(f)

# ----------------------------
# initialize graph
# ----------------------------

g = Graph()

EX = Namespace("http://example.org/")
UNIPROT = Namespace("http://purl.uniprot.org/uniprot/")
CHEMBL_MOL = Namespace("https://www.ebi.ac.uk/chembl/compound/")
CHEMBL_TARGET = Namespace("https://www.ebi.ac.uk/chembl/target_report_card/")
REACTOME = Namespace("https://reactome.org/content/detail/")
EFO = Namespace("http://www.ebi.ac.uk/efo/")
DOID = Namespace("http://purl.obolibrary.org/obo/DOID_")

g.bind("ex", EX)
g.bind("uniprot", UNIPROT)
g.bind("chemblmol", CHEMBL_MOL)
g.bind("chembltarget", CHEMBL_TARGET)
g.bind("reactome", REACTOME)
g.bind("efo", EFO)
g.bind("doid", DOID)
g.bind("rdfs", RDFS)
g.bind("skos", SKOS)

# ----------------------------
# ontology / schema hints
# ----------------------------

g.add((EX.Drug, RDF.type, RDFS.Class))
g.add((EX.Protein, RDF.type, RDFS.Class))
g.add((EX.Pathway, RDF.type, RDFS.Class))
g.add((EX.Disease, RDF.type, RDFS.Class))
g.add((EX.Target, RDF.type, RDFS.Class))

g.add((EX.Drug, RDFS.label, Literal("Drug")))
g.add((EX.Protein, RDFS.label, Literal("Protein")))
g.add((EX.Pathway, RDFS.label, Literal("Pathway")))
g.add((EX.Disease, RDFS.label, Literal("Disease")))
g.add((EX.Target, RDFS.label, Literal("Target")))

g.add((EX.targets, RDF.type, RDF.Property))
g.add((EX.associatedWith, RDF.type, RDF.Property))
g.add((EX.participatesIn, RDF.type, RDF.Property))
g.add((EX.actsOnTarget, RDF.type, RDF.Property))
g.add((EX.actionType, RDF.type, RDF.Property))
g.add((EX.mechanismOfAction, RDF.type, RDF.Property))
g.add((EX.associationScore, RDF.type, RDF.Property))
g.add((EX.uniprotAccession, RDF.type, RDF.Property))

g.add((EX.targets, RDFS.domain, EX.Drug))
g.add((EX.targets, RDFS.range, EX.Protein))

g.add((EX.associatedWith, RDFS.domain, EX.Protein))
g.add((EX.associatedWith, RDFS.range, EX.Disease))

g.add((EX.participatesIn, RDFS.domain, EX.Protein))
g.add((EX.participatesIn, RDFS.range, EX.Pathway))

g.add((EX.actsOnTarget, RDFS.domain, EX.Drug))
g.add((EX.actsOnTarget, RDFS.range, EX.Target))

# ----------------------------
# pathway
# ----------------------------

pathway_id = "R-HSA-5673001"
pathway_name = "RAF/MAP kinase cascade"
pathway_uri = REACTOME[pathway_id]

g.add((pathway_uri, RDF.type, EX.Pathway))
g.add((pathway_uri, RDFS.label, Literal(pathway_name)))

# ----------------------------
# protein aliases
# ----------------------------

protein_aliases = {
    "EGFR": ["ERBB1", "HER1"],
    "BRAF": ["B-RAF"],
    "MAPK1": ["ERK2"],
    "MAPK3": ["ERK1"]
}

# ----------------------------
# disease aliases
# ----------------------------

disease_aliases = {
    "melanoma": ["skin melanoma", "cutaneous melanoma"]
}

# ----------------------------
# proteins
# ----------------------------

for p in proteins:

    accession = p["uniprot_accession"]
    full_name = p["name"]

    short_name = full_name.split()[-1] if full_name else accession

    protein_uri = UNIPROT[accession]

    g.add((protein_uri, RDF.type, EX.Protein))
    g.add((protein_uri, RDFS.label, Literal(short_name)))
    g.add((protein_uri, EX.uniprotAccession, Literal(accession)))
    g.add((protein_uri, EX.participatesIn, pathway_uri))

    if short_name in protein_aliases:
        for alias in protein_aliases[short_name]:
            g.add((protein_uri, SKOS.altLabel, Literal(alias)))

# ----------------------------
# drugs
# ----------------------------

for row in chembl:

    accession = row["uniprot_accession"]
    mol_id = row["molecule_chembl_id"]
    drug_name = row.get("drug_name")
    action_type = row.get("action_type")
    mechanism = row.get("mechanism_of_action")
    target_id = row.get("target_chembl_id")
    target_name = row.get("target_name")

    if not mol_id:
        continue

    protein_uri = UNIPROT[accession]
    drug_uri = CHEMBL_MOL[mol_id]

    g.add((drug_uri, RDF.type, EX.Drug))
    g.add((drug_uri, EX.targets, protein_uri))

    if drug_name:
        g.add((drug_uri, RDFS.label, Literal(drug_name)))

    if action_type:
        g.add((drug_uri, EX.actionType, Literal(action_type)))

    if mechanism:
        g.add((drug_uri, EX.mechanismOfAction, Literal(mechanism)))

    if target_id:
        target_uri = CHEMBL_TARGET[target_id]
        g.add((target_uri, RDF.type, EX.Target))
        g.add((drug_uri, EX.actsOnTarget, target_uri))

        if target_name:
            g.add((target_uri, RDFS.label, Literal(target_name)))

# ----------------------------
# diseases
# ----------------------------

for row in diseases:

    accession = row["uniprot_accession"]
    disease_id = row.get("disease_id")
    disease_name = row.get("disease_name")
    score = row.get("association_score")

    if not disease_id:
        continue

    protein_uri = UNIPROT[accession]

    if disease_id.startswith("EFO_"):
        disease_uri = EFO[disease_id]
    elif disease_id.startswith("DOID_"):
        disease_uri = DOID[disease_id.replace("DOID_", "")]
    else:
        disease_uri = EX[disease_id]

    g.add((disease_uri, RDF.type, EX.Disease))
    g.add((protein_uri, EX.associatedWith, disease_uri))

    if disease_name:
        g.add((disease_uri, RDFS.label, Literal(disease_name)))

        key = disease_name.lower()
        if key in disease_aliases:
            for alias in disease_aliases[key]:
                g.add((disease_uri, SKOS.altLabel, Literal(alias)))

    if score is not None:
        g.add((protein_uri, EX.associationScore, Literal(score)))

# ----------------------------
# output
# ----------------------------

Path("data_processed").mkdir(exist_ok=True)

triples_path = "data_processed/graph_triples.tsv"

with open(triples_path, "w") as f:
    for s, p, o in g:
        f.write(f"{s}\t{p}\t{o}\n")

print(f"Saved raw triples to {triples_path}")

output_path = "data_processed/graph.ttl"
g.serialize(destination=output_path, format="turtle")

print(f"Saved RDF graph to {output_path}")
print(f"Total triples: {len(g)}")

## Lets have a look at the raw RDF triples that make up the knowledge graph

In [ ]:
import pandas as pd

df = pd.read_csv("data_processed/graph_triples.tsv", sep="\t", names=["subject","predicate","object"])
df.head(10)

## Now let's have a look at the turtle (ttl) formatted version of the triples

This is the file we will use to create the knowledge graph.  

In [ ]:
with open("data_processed/graph.ttl") as f:
    lines=f.readlines()
print(lines[:20])


# Talk to Your Graph in Python

This notebook loads the RDF graph, runs a few hand-written SPARQL queries, and then adds a **question → SPARQL → RDFLib query → natural-language answer** loop.

## Prerequisites

- `data_processed/graph.ttl` exists
- `OPENAI_API_KEY` is set in your environment
- Python packages installed:
  - `rdflib`
  - `openai`


In [ ]:
import sys
print(sys.executable)

In [ ]:
from pathlib import Path
from rdflib import Graph

GRAPH_PATH = Path("data_processed/graph.ttl")
assert GRAPH_PATH.exists(), f"Could not find {GRAPH_PATH}"

g = Graph()
g.parse(GRAPH_PATH, format="turtle")
print(f"Loaded triples: {len(g):,}")

In [ ]:
PREFIXES = """
PREFIX ex: <http://example.org/>
PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
PREFIX chemblmol: <https://www.ebi.ac.uk/chembl/compound/>
PREFIX chembltarget: <https://www.ebi.ac.uk/chembl/target_report_card/>
PREFIX reactome: <https://reactome.org/content/detail/>
PREFIX efo: <http://www.ebi.ac.uk/efo/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
"""

print(PREFIXES)

# Build a small subgraph to test visualization

Let's select a small subset of the full graph for simplicity and clarity

In [ ]:
import networkx as nx

G = nx.DiGraph()

query = PREFIXES + """
SELECT ?drugLabel ?proteinLabel
WHERE {
  ?drug ex:targets ?protein ;
        rdfs:label ?drugLabel .
  ?protein rdfs:label ?proteinLabel .
}
LIMIT 25
"""

for drug, protein in g.query(query):
    G.add_node(drug, type="drug")
    G.add_node(protein, type="protein")
    G.add_edge(drug, protein)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Visualize the subgraph

This will generate a small interactive subgraph showing Drug → Protein 

Proteins = BLUE and 
Drugs = RED

In [ ]:
from pyvis.network import Network

net = Network(notebook=True, directed=True)

for node in G.nodes:
    node_type = G.nodes[node]["type"]
    color = "red" if node_type == "drug" else "blue"
    net.add_node(node, label=node, color=color)

for source, target in G.edges:
    net.add_edge(source, target)

net.show("drug_target_graph.html")

## A few direct SPARQL examples

Now you've seen how visualization can be achieved, let's go back to the full graph to do some querying.
First we'll try some SPARQL queries and then we'll move onto querying using natural language via an LLM call.

In [ ]:
q1 = PREFIXES + """
SELECT ?proteinLabel
WHERE {
  ?protein ex:participatesIn reactome:R-HSA-5673001 ;
           rdfs:label ?proteinLabel .
}
ORDER BY ?proteinLabel
LIMIT 20
"""

for row in g.query(q1):
    print(row[0])

In [ ]:
q2 = PREFIXES + """
SELECT ?drugLabel
WHERE {
  ?drug ex:targets ?protein ;
        rdfs:label ?drugLabel .
  ?protein rdfs:label|skos:altLabel "EGFR" .
}
ORDER BY ?drugLabel
"""

for row in g.query(q2):
    print(row[0])

In [ ]:
q3 = PREFIXES + """
SELECT ?drugLabel ?diseaseLabel
WHERE {
  ?drug ex:targets ?protein ;
        rdfs:label ?drugLabel .
  ?protein ex:associatedWith ?disease .
  ?disease rdfs:label|skos:altLabel "melanoma" ;
           rdfs:label ?diseaseLabel .
}
ORDER BY ?drugLabel
LIMIT 30
"""

for row in g.query(q3):
    print(row)

## LLM-assisted querying

The function below asks an LLM to produce a SPARQL query in JSON, runs that query with RDFLib, and returns the rows.

It is explicitly told to use:

- `rdfs:label|skos:altLabel` for entity lookup
- the graph predicates you created earlier


In [ ]:
import os, json
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

SCHEMA = """
You are writing SPARQL for a small biomedical RDF graph.

Use these prefixes exactly:

PREFIX ex: <http://example.org/>
PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
PREFIX chemblmol: <https://www.ebi.ac.uk/chembl/compound/>
PREFIX chembltarget: <https://www.ebi.ac.uk/chembl/target_report_card/>
PREFIX reactome: <https://reactome.org/content/detail/>
PREFIX efo: <http://www.ebi.ac.uk/efo/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

Classes:
- ex:Drug
- ex:Protein
- ex:Pathway
- ex:Disease
- ex:Target

Main predicates:
- ex:targets (Drug -> Protein)
- ex:associatedWith (Protein -> Disease)
- ex:participatesIn (Protein -> Pathway)
- ex:actsOnTarget (Drug -> Target)

Useful labels:
- rdfs:label = preferred label
- skos:altLabel = alias/synonym

Important rule:
When looking up a named entity like EGFR, ERBB1, HER1, melanoma, skin melanoma, or cutaneous melanoma,
use this pattern:
    ?entity rdfs:label|skos:altLabel ?name .
Do not rely only on rdfs:label.

Main pathway:
- reactome:R-HSA-5673001 = RAF/MAP kinase cascade

Return only a SELECT query.
"""

def generate_sparql(question: str) -> str:
    prompt = f"""{SCHEMA}

Question: {question}
"""

    resp = client.responses.create(
        model="gpt-4o",
        input=prompt,
        temperature=0,
        text={
            "format": {
                "type": "json_schema",
                "name": "sparql_response",
                "schema": {
                    "type": "object",
                    "properties": {
                        "sparql": {"type": "string"}
                    },
                    "required": ["sparql"],
                    "additionalProperties": False
                }
            }
        }
    )
    payload = json.loads(resp.output_text)
    return payload["sparql"]

In [ ]:
def run_sparql(query: str, max_rows: int = 20):
    rows = list(g.query(query))
    print(f"Rows returned: {len(rows)}")
    for row in rows[:max_rows]:
        print(row)
    return rows

In [ ]:
def ask_graph(question: str, summarize: bool = True):
    sparql = generate_sparql(question)
    print("SPARQL:\n")
    print(sparql)
    print("\n---\n")
    rows = list(g.query(sparql))
    print(f"Rows returned: {len(rows)}")
    for row in rows[:10]:
        print(row)

    if not summarize:
        return {"question": question, "sparql": sparql, "rows": rows}

    summary_prompt = f"""Answer the user's question using only these query results.

Question:
{question}

SPARQL:
{sparql}

Rows:
{rows[:25]}

Give a brief answer. If there are no rows, say that no matching results were found in this graph.
"""

    resp = client.responses.create(
        model="gpt-4o",
        input=summary_prompt,
        temperature=0.2
    )
    answer = resp.output_text
    print("\nAnswer:\n")
    print(answer)
    return {"question": question, "sparql": sparql, "rows": rows, "answer": answer}

## Example questions

In [ ]:
ask_graph("Which drugs target EGFR?")

In [ ]:
ask_graph("Which drugs target ERBB1?")

In [ ]:
ask_graph("Which diseases are associated with cutaneous melanoma?")

In [ ]:
ask_graph("Which drugs target proteins associated with melanoma?")

### Congratulations!
### You have now:
### - Encountered real graph data sources and file formats
### - Built your own biological knowledgebase from real data
### - Visualized graph structure
### - Queried your graph using SPARQL
### - Used an LLM to query your graph in natural language

## - Now try adding a new annotation e.g. gene → phenoype